# Agent Observability — Traces, Spans, and Telemetry
## Traces, Spans, Events — Making Agents Inspectable
⏱ ~12 min

**Observability** is the ability to understand what your agent is doing from the outside — without changing its source code. For LLM agents this means capturing *when* each model call happened, *how long* it took, *how many tokens* it consumed, *what errors* occurred, and *what the inputs and outputs were*.

By the end of this session you will understand the observability primitives (traces, spans, events), know how to instrument any LangGraph agent with a custom `BaseCallbackHandler`, and be able to compare the zero-dependency local approach with hosted platforms like LangSmith and OpenTelemetry.

---

### Workshop Roadmap

| # | Topic |
|---|-------|
| 1 | **Concepts** — Traces, spans, events, and why they matter |
| 2 | **Callback Lifecycle** — `on_llm_start`, `on_llm_end`, `on_llm_error` |
| 3 | **Custom Tracer** — Build `ObservabilityCallback` from scratch |
| 4 | **Multi-Node Tracing** — Track LangGraph chains and tool calls |
| 5 | **Structured Logging** — Emit JSONL traces to disk |
| 6 | **LangSmith Integration** — One env-var toggle for hosted traces |
| ★ | **Advanced** — Cost estimation, dashboards, error injection |

---

### Prerequisites
- Python 3.10+ with the repo `.venv` activated (all deps in `requirements.txt`)
- `OPENAI_API_KEY` in `.env` (or Colab Secrets)
- Optional: `LANGSMITH_API_KEY` for Part 6 (free account at smith.langchain.com)

### Key References
> OpenTelemetry specification: https://opentelemetry.io/docs/concepts/signals/traces/  
> LangChain Callbacks guide: https://python.langchain.com/docs/concepts/callbacks/  
> LangSmith Tracing docs: https://docs.smith.langchain.com/observability/  
> OpenLLMetry (OTEL for LLMs): https://github.com/traceloop/openllmetry  

In [ ]:
# Install dependencies — runs only on Google Colab.
# Local users: your .venv already has everything from requirements.txt.
import sys


def _in_colab():
    try:
        import google.colab

        return True
    except ImportError:
        return False


if _in_colab():
    import subprocess

    subprocess.run(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            "langchain",
            "langchain-openai",
            "langgraph",
            "python-dotenv",
        ]
    )
    print("Colab install complete.")
else:
    print("Local environment detected — skipping install.")

In [ ]:
import inspect
import json
import os
import time
from pathlib import Path
from typing import Any

# ── API key ───────────────────────────────────────────────────────────────────
# Colab: set in Secrets panel (left sidebar key icon)
# Local: create a .env file containing  OPENAI_API_KEY=sk-...
try:
    from google.colab import userdata

    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
except ImportError:
    from dotenv import load_dotenv

    load_dotenv()

# ── Core imports ──────────────────────────────────────────────────────────────
from langchain.callbacks.base import BaseCallbackHandler
from langchain_core.messages import HumanMessage
from langchain_core.outputs import LLMResult
from langchain_openai import ChatOpenAI
from langgraph.graph import END, StateGraph
from typing_extensions import TypedDict

# ── Sanity check ──────────────────────────────────────────────────────────────
key = os.environ.get("OPENAI_API_KEY", "")
key_ok = bool(key) and key.startswith("sk-")
print(f"OPENAI_API_KEY present and valid: {key_ok}")
if not key_ok:
    print()
    print("  ACTION REQUIRED — add your key before running LLM cells.")
    print("  Local: echo 'OPENAI_API_KEY=sk-...' >> .env")
    print("  Colab: Secrets panel → Add secret 'OPENAI_API_KEY'")

---

## Part 1 — What Is Agent Observability and Why Does It Exist?

---

### The Problem

LLM agents are opaque by default. When something goes wrong — a slow response, a wrong answer, an unexpected tool call — you have no way to answer:

- Which LLM call took 8 seconds?
- Which prompt used 3 000 tokens when it should have used 300?
- Did the agent choose the wrong tool, or did the tool fail?
- Which user request triggered the most expensive path?

Without instrumentation, every bug hunt starts from scratch.

---

### The Observability Stack

```
┌─────────────────────────────────────────────────────────┐
│                    YOUR AGENT                           │
│                                                         │
│  ┌─────────┐    ┌──────────┐    ┌───────────────────┐  │
│  │ LangGraph│───▶│  LLM     │───▶│  Tool / Retriever │  │
│  │  node   │    │  call    │    │  call             │  │
│  └─────────┘    └──────────┘    └───────────────────┘  │
│        │               │                  │             │
│        ▼               ▼                  ▼             │
│  ┌──────────────────────────────────────────────────┐  │
│  │          CALLBACK HANDLER (tracer)               │  │
│  │  on_chain_start   on_llm_start   on_tool_start   │  │
│  │  on_chain_end     on_llm_end     on_tool_end     │  │
│  │  on_chain_error   on_llm_error   on_tool_error   │  │
│  └────────────────────┬─────────────────────────────┘  │
└───────────────────────┼─────────────────────────────────┘
                        │
           ┌────────────▼──────────────┐
           │     SPAN / TRACE DATA     │
           │  run_id, latency_ms,      │
           │  prompt_tokens,           │
           │  completion_tokens,       │
           │  error, input, output     │
           └────────────┬──────────────┘
                        │
           ┌────────────▼──────────────┐
           │       BACKEND             │
           │  ┌──────────────────────┐ │
           │  │  In-memory list      │ │  ← this notebook
           │  │  JSONL log file      │ │  ← Part 5
           │  │  LangSmith           │ │  ← Part 6
           │  │  OpenTelemetry/OTLP  │ │  ← Bonus
           │  └──────────────────────┘ │
           └───────────────────────────┘
```

> **Source**: Architecture adapted from the OpenTelemetry specification (CNCF, 2024) and LangChain Callbacks documentation.

---

### Key Vocabulary

| Term | Definition |
|------|------------|
| **Trace** | The full end-to-end record of one user request — from input to final output |
| **Span** | One unit of work within a trace (e.g., a single LLM call, a tool call) |
| **Event** | A timestamped annotation attached to a span (e.g., "retrieved 3 chunks") |
| **run_id** | A UUID that uniquely identifies one span — correlates `start` and `end` callbacks |
| **parent_run_id** | Links child spans (LLM calls) to their parent span (LangGraph node) |
| **Latency** | Wall-clock time from `on_llm_start` to `on_llm_end` in milliseconds |
| **Token usage** | `prompt_tokens` + `completion_tokens` from `response.llm_output["token_usage"]` |
| **Cost** | Estimated spend = tokens × price-per-token for the model tier |

---

### Three Approaches Compared

| Approach | Setup | Storage | UI | Cost | Best for |
|----------|-------|---------|----|----|----------|
| **Custom callback** (this notebook) | Zero deps | In-memory / JSONL | None | Free | Local dev, learning, no external accounts |
| **LangSmith** | 3 env vars + account | Hosted (LangChain cloud) | Full web UI | Free tier | Team debugging, evals, prompt management |
| **OpenTelemetry** | OTLP exporter + collector | Any OTEL backend | Depends on backend | Backend cost | Production, multi-service, existing infra |
| **Arize Phoenix** | `pip install arize-phoenix` | Local SQLite or hosted | Local web UI | Free OSS | Evaluation-focused observability |

---

## Part 2 — The Callback Lifecycle

---

### How LangChain Fires Events

LangChain's callback system is an event bus. Every time a component runs, it fires lifecycle hooks. You attach a `BaseCallbackHandler` subclass to intercept them.

```
llm.invoke([message], config={"callbacks": [my_handler]})
                                                │
               ┌────────────────────────────────┘
               │
               ▼
  on_llm_start(serialized, prompts, run_id, **kwargs)
        │  called BEFORE the HTTP request is sent to the model
        │  → record time.time() keyed by str(run_id)
        │
        ▼  [model processes request...]
        │
  on_llm_end(response, run_id, **kwargs)
        │  called AFTER the model returns
        │  → pop start time, compute elapsed ms
        │  → read response.llm_output["token_usage"]
        │
        ▼  (or, if something went wrong)
        │
  on_llm_error(error, run_id, **kwargs)
        │  called if the LLM raises ANY exception
        │  → record error string, do NOT re-raise
```

The same pattern applies to chains and tools:

| Event group | Start hook | End hook | Error hook |
|-------------|-----------|----------|------------|
| LLM calls | `on_llm_start` | `on_llm_end` | `on_llm_error` |
| Chain / node | `on_chain_start` | `on_chain_end` | `on_chain_error` |
| Tool calls | `on_tool_start` | `on_tool_end` | `on_tool_error` |
| Retriever | `on_retriever_start` | `on_retriever_end` | `on_retriever_error` |

### Why `run_id`?

In a concurrent system, multiple LLM calls can be in-flight simultaneously. Each call gets a unique `run_id` UUID — so the `end` event for call B does not accidentally get matched to the `start` event for call A. We use a `dict[str, float]` keyed by `str(run_id)` to track open spans.

In [ ]:
# ─── 2-A: Inspect the BaseCallbackHandler interface ──────────────────────────
# Before building our own handler, see what lifecycle methods are available.

# Implement only the hooks you need; the base class no-ops the rest.
# Every lifecycle event — LLM, chain, tool, retriever — has a start/end/error hook.
hooks = [
    name
    for name, _ in inspect.getmembers(BaseCallbackHandler, predicate=inspect.isfunction)
    if name.startswith("on_")
]

print(f"BaseCallbackHandler exposes {len(hooks)} lifecycle hooks:\n")
for h in sorted(hooks):
    print(f"  {h}")

In [ ]:
# ─── 2-B: Minimal tracing callback — see events fire in real time ─────────────
# This handler just PRINTS each event so you can watch the lifecycle unfold.
# Use this pattern to debug any callback implementation.


# Inheriting BaseCallbackHandler and overriding on_llm_* is the minimum viable tracer.
class PrintCallbackHandler(BaseCallbackHandler):
    """Prints every LLM lifecycle event to stdout. Educational use only."""

    def on_llm_start(self, serialized: dict, prompts: list, run_id: Any, **kwargs):
        print(f"[START] run_id={str(run_id)[:8]}...")
        print(f"        model={serialized.get('kwargs', {}).get('model_name', '?')}")
        print(f"        prompt_preview={repr(prompts[0][:80])}...")

    def on_llm_end(self, response: LLMResult, run_id: Any, **kwargs):
        usage = response.llm_output.get("token_usage", {}) if response.llm_output else {}
        print(f"[END]   run_id={str(run_id)[:8]}...")
        print(f"        prompt_tokens={usage.get('prompt_tokens', '?')}")
        print(f"        completion_tokens={usage.get('completion_tokens', '?')}")
        output = response.generations[0][0].text[:80] if response.generations else ""
        print(f"        output_preview={repr(output)}...")

    def on_llm_error(self, error: Any, run_id: Any, **kwargs):
        print(f"[ERROR] run_id={str(run_id)[:8]}... error={error}")


# Attach and fire a real LLM call
print_cb = PrintCallbackHandler()
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
# temperature=0 → deterministic responses, easier to compare across runs.

print("=== Running LLM call with PrintCallbackHandler ===\n")
response = llm.invoke(
    [HumanMessage(content="What is a span in distributed tracing? One sentence.")],
    config={"callbacks": [print_cb]},
)
print(f"\nFinal answer: {response.content}")

---

## Part 3 — Build the ObservabilityCallback

---

Now we build the real tracer: a stateful handler that accumulates span records across all calls in a session, computes summary metrics, and exposes a `report()` method.

### Design Decisions

| Decision | Choice | Rationale |
|----------|--------|-----------|
| Storage | `list[dict]` in-memory | Simple, zero-dep, easy to inspect |
| Run correlation | `dict[str, tuple]` keyed by `str(run_id)` | UUID keys; stores `(start_time, model_name)` per open span |
| Error handling | Record error string, do NOT raise | Observability code must not crash the agent |
| Token extraction | `response.llm_output["token_usage"]` | OpenAI-specific; falls back to 0 for other providers |
| Cost estimation | `prompt_tokens * prompt_rate + completion_tokens * completion_rate` | Per-model rate table, easily extensible |

### Token Pricing Reference (June 2025)

| Model | Prompt (per 1M tokens) | Completion (per 1M tokens) |
|-------|----------------------|---------------------------|
| `gpt-4o-mini` | $0.15 | $0.60 |
| `gpt-4o` | $2.50 | $10.00 |
| `gpt-4.1` | $2.00 | $8.00 |
| `gpt-4.1-mini` | $0.40 | $1.60 |

> Always check https://openai.com/api/pricing for the latest rates — this table can go stale.

In [ ]:
# ─── 3-A: Full ObservabilityCallback with cost estimation ─────────────────────

# Per-million-token pricing (June 2025 — update if prices change)
MODEL_PRICING: dict[str, dict[str, float]] = {
    "gpt-4o-mini": {"prompt": 0.15, "completion": 0.60},
    "gpt-4o": {"prompt": 2.50, "completion": 10.00},
    "gpt-4.1": {"prompt": 2.00, "completion": 8.00},
    "gpt-4.1-mini": {"prompt": 0.40, "completion": 1.60},
}


class ObservabilityCallback(BaseCallbackHandler):
    """Stateful tracer: records per-call latency, token counts, cost, and errors."""

    def __init__(self):
        # Accumulated span records — one dict per completed LLM call
        self.calls: list[dict] = []
# run_id uniquely identifies each invocation — safe for concurrent/async calls.
        # Open spans keyed by run_id — stores (start_time, model_name)
        self._starts: dict[str, tuple[float, str]] = {}

    # ── Lifecycle hooks ──────────────────────────────────────────────────────

    def on_llm_start(
        self, serialized: dict, prompts: list, run_id: Any, **kwargs
    ):
        model = serialized.get("kwargs", {}).get("model_name", "unknown")
        self._starts[str(run_id)] = (time.time(), model)

    def on_llm_end(self, response: LLMResult, run_id: Any, **kwargs):
        start_time, model = self._starts.pop(str(run_id), (time.time(), "unknown"))
        latency_ms = round((time.time() - start_time) * 1000)

        # Extract token usage from OpenAI response metadata
        usage: dict = {}
        if response.llm_output:
            usage = response.llm_output.get("token_usage", {})

        prompt_tokens = usage.get("prompt_tokens", 0)
        completion_tokens = usage.get("completion_tokens", 0)

        # Estimate cost using per-model pricing table
        pricing = MODEL_PRICING.get(model, {"prompt": 0.0, "completion": 0.0})
        cost_usd = (
            prompt_tokens * pricing["prompt"] / 1_000_000
            + completion_tokens * pricing["completion"] / 1_000_000
        )

        self.calls.append(
            {
                "run_id": str(run_id),
                "model": model,
                "latency_ms": latency_ms,
                "prompt_tokens": prompt_tokens,
                "completion_tokens": completion_tokens,
                "total_tokens": prompt_tokens + completion_tokens,
                "cost_usd": round(cost_usd, 6),
            }
        )

    def on_llm_error(self, error: Any, run_id: Any, **kwargs):
# Must NOT re-raise: a crash in on_llm_error would swallow the agent's error handling.
        # Record the error WITHOUT re-raising — observability code must not crash the agent
        self._starts.pop(str(run_id), None)
        self.calls.append(
            {
                "run_id": str(run_id),
                "error": str(error),
            }
        )

    # ── Aggregation ──────────────────────────────────────────────────────────

    def report(self) -> dict:
        """Return a summary dict across all completed calls in this session."""
        ok = [c for c in self.calls if "error" not in c]
        errors = [c for c in self.calls if "error" in c]
        if not ok:
            return {"total_calls": 0, "errors": len(errors), "message": "no successful calls"}
        return {
            "total_calls": len(self.calls),
            "errors": len(errors),
            "avg_latency_ms": round(sum(c["latency_ms"] for c in ok) / len(ok)),
            "max_latency_ms": max(c["latency_ms"] for c in ok),
            "total_prompt_tokens": sum(c["prompt_tokens"] for c in ok),
            "total_completion_tokens": sum(c["completion_tokens"] for c in ok),
            "total_tokens": sum(c["total_tokens"] for c in ok),
            "total_cost_usd": round(sum(c["cost_usd"] for c in ok), 6),
        }


print("ObservabilityCallback defined.")
print("Hooks implemented:", [m for m in dir(ObservabilityCallback) if m.startswith("on_")])

In [ ]:
# ─── 3-B: Run multiple calls and inspect per-call records ─────────────────────

DEMO_TASKS = [
    "What is a trace in distributed systems? One sentence.",
    "What is a span in OpenTelemetry? One sentence.",
    "Name two metrics every LLM agent should track. Be brief.",
]

cb = ObservabilityCallback()
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

for task in DEMO_TASKS:
    result = llm.invoke(
        [HumanMessage(content=task)],
        config={"callbacks": [cb]},
    )
    print(f"Task: {task}")
    print(f"  Answer: {result.content}")
    print()

print("=== Per-call records ===")
for i, call in enumerate(cb.calls):
    print(
        f"  Call {i}: latency={call['latency_ms']}ms  "
        f"tokens={call['total_tokens']}  "
        f"cost=${call['cost_usd']:.6f}"
    )

In [ ]:
# ─── 3-C: Print the aggregate report ─────────────────────────────────────────

report = cb.report()
print("=== Telemetry Report ===")
for k, v in report.items():
    print(f"  {k}: {v}")

print()
cost_per_call = report["total_cost_usd"] / max(report["total_calls"], 1)
print(f"Cost per call (avg): ${cost_per_call:.6f}")

total = max(report["total_tokens"], 1)
completion_pct = 100 * report["total_completion_tokens"] / total
print(
    f"Token efficiency: {report['total_completion_tokens']} completion "
    f"/ {report['total_tokens']} total = {completion_pct:.1f}% output"
)

---

## Part 4 — Multi-Node Tracing with LangGraph

---

Real agents are LangGraph graphs with multiple nodes. We want to trace:
- Each LLM call (already done with `on_llm_start` / `on_llm_end`)
- Each **graph node** (chain) — to see which node was slow
- Parent-child relationships — which node triggered which LLM call

### Traces vs Spans in a LangGraph Context

```
TRACE: user request "summarize this article"
  │
  ├── SPAN: graph.invoke()                  [on_chain_start / on_chain_end]
  │     ├── SPAN: node "fetch_article"      [on_chain_start / on_chain_end]
  │     │     └── SPAN: tool http_get()     [on_tool_start / on_tool_end]
  │     └── SPAN: node "summarize"          [on_chain_start / on_chain_end]
  │           └── SPAN: llm.invoke()        [on_llm_start / on_llm_end]
  │                 latency=1240ms
  │                 prompt_tokens=812
  │                 completion_tokens=94
  └── total latency: 1350ms
```

> `parent_run_id` in each callback links child spans to their parent — this is how LangSmith builds the trace tree view.

In [ ]:
# ─── 4-A: Extend the callback to also track chain (node) timings ──────────────


class FullTracer(ObservabilityCallback):
    """Extends ObservabilityCallback with chain-level span tracking."""

    def __init__(self):
        super().__init__()
        self.chain_spans: list[dict] = []
        self._chain_starts: dict[str, float] = {}

# In LangGraph each node is a chain — on_chain_start fires once per node execution.
    def on_chain_start(
        self,
        serialized: dict,
        inputs: dict,
        run_id: Any,
        parent_run_id: Any = None,
        **kwargs,
    ):
        self._chain_starts[str(run_id)] = time.time()

    def on_chain_end(
        self,
        outputs: dict,
        run_id: Any,
        parent_run_id: Any = None,
        **kwargs,
    ):
        start = self._chain_starts.pop(str(run_id), time.time())
        self.chain_spans.append(
            {
                "run_id": str(run_id),
                "parent_run_id": str(parent_run_id) if parent_run_id else None,
# parent_run_id links child spans to their parent — this is how you reconstruct the call tree.
                "latency_ms": round((time.time() - start) * 1000),
            }
        )

    def on_chain_error(
        self,
        error: Any,
        run_id: Any,
        parent_run_id: Any = None,
        **kwargs,
    ):
        self._chain_starts.pop(str(run_id), None)
        self.chain_spans.append(
            {
                "run_id": str(run_id),
                "parent_run_id": str(parent_run_id) if parent_run_id else None,
                "error": str(error),
            }
        )

    def chain_report(self) -> dict:
        ok = [s for s in self.chain_spans if "error" not in s]
        if not ok:
            return {"chain_spans": 0}
        return {
            "chain_spans": len(ok),
            "avg_chain_latency_ms": round(sum(s["latency_ms"] for s in ok) / len(ok)),
            "max_chain_latency_ms": max(s["latency_ms"] for s in ok),
        }


print("FullTracer defined — tracks LLM spans AND chain (node) spans.")

In [ ]:
# ─── 4-B: Build a two-node LangGraph and trace it ────────────────────────────
# Node 1 (draft): generate a first-pass answer
# Node 2 (refine): improve the draft
# Both nodes make LLM calls — we should see 2 LLM spans per request.


# TypedDict gives LangGraph typed access to state keys with zero runtime overhead.
class AgentState(TypedDict):
    task: str
    draft: str
    response: str


def create_two_node_workflow(tracer: FullTracer):
    llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.3)
    graph = StateGraph(AgentState)

    def draft_node(state: AgentState) -> AgentState:
        """Node 1: Generate a draft answer."""
        result = llm.invoke(
            [HumanMessage(content=f"Draft a short answer to: {state['task']}")],
            config={"callbacks": [tracer]},
        )
        return {"draft": result.content}

    def refine_node(state: AgentState) -> AgentState:
        """Node 2: Improve the draft answer."""
        result = llm.invoke(
            [
                HumanMessage(
                    content=f"Improve this answer to be clearer and more concise:\n{state['draft']}"
                )
            ],
            config={"callbacks": [tracer]},
        )
        return {"response": result.content}

    graph.add_node("draft", draft_node)
    graph.add_node("refine", refine_node)
    graph.set_entry_point("draft")
    graph.add_edge("draft", "refine")
    graph.add_edge("refine", END)
    return graph.compile()
# compile() locks the graph topology — after this the node set and edges are immutable.


tracer = FullTracer()
app = create_two_node_workflow(tracer)

result = app.invoke(
    {"task": "Explain what observability means for LLM agents", "draft": "", "response": ""}
)

print("=== Final Response ===")
print(result["response"])
print()

print("=== LLM Spans ===")
for i, call in enumerate(tracer.calls):
    print(f"  LLM call {i}: latency={call['latency_ms']}ms  tokens={call['total_tokens']}")

print()
print("=== LLM Aggregate ===")
for k, v in tracer.report().items():
    print(f"  {k}: {v}")

In [ ]:
# ─── 4-C: Trace multiple requests and compare per-call stats ─────────────────

tasks = [
    "What is latency in the context of LLM APIs?",
    "What is a token in the context of language models?",
    "Explain the difference between a trace and a span.",
]

multi_tracer = FullTracer()
multi_app = create_two_node_workflow(multi_tracer)

for task in tasks:
    multi_app.invoke({"task": task, "draft": "", "response": ""})

n_calls = len(multi_tracer.calls)
print(f"Ran {len(tasks)} tasks × 2 nodes = {len(tasks) * 2} expected LLM calls")
print(f"Actual LLM spans recorded: {n_calls}\n")

header = f"  {'#':<4} {'latency_ms':<14} {'prompt_tok':<14} {'completion_tok':<18} {'cost_usd':<12}"
print(header)
print("  " + "-" * 62)
for i, call in enumerate(multi_tracer.calls):
    print(
        f"  {i:<4} {call['latency_ms']:<14} {call['prompt_tokens']:<14} "
        f"{call['completion_tokens']:<18} {call['cost_usd']:<12}"
    )

print()
rpt = multi_tracer.report()
print(f"Total cost: ${rpt['total_cost_usd']:.6f}")
print(f"Avg latency: {rpt['avg_latency_ms']}ms")

---

## Part 5 — Structured Logging: JSONL Traces to Disk

---

An in-memory list is useful for a single session, but it disappears when the process ends. **JSONL** (JSON Lines) is the standard format for structured log files: one JSON object per line, easy to parse, easy to stream.

```
traces.jsonl:
{"run_id": "abc...", "model": "gpt-4o-mini", "latency_ms": 830, "total_tokens": 102, ...}
{"run_id": "def...", "model": "gpt-4o-mini", "latency_ms": 1210, "total_tokens": 198, ...}
{"run_id": "xyz...", "model": "gpt-4o-mini", "latency_ms": 660, "total_tokens": 87, ...}
```

This format works with any log aggregator (Loki, Elasticsearch, S3 + Athena, Datadog) without any schema migration — just append lines and query later.

### JSONL Record Schema

| Field | Type | Description |
|-------|------|-------------|
| `run_id` | `str` (UUID) | Unique span identifier |
| `model` | `str` | Model name from serialized callback metadata |
| `latency_ms` | `int` | Wall-clock ms from `on_llm_start` to `on_llm_end` |
| `prompt_tokens` | `int` | Tokens in the input prompt |
| `completion_tokens` | `int` | Tokens in the model's output |
| `total_tokens` | `int` | `prompt_tokens + completion_tokens` |
| `cost_usd` | `float` | Estimated cost in USD |
| `timestamp` | `str` | ISO 8601 UTC timestamp at log-write time |

In [ ]:
# ─── 5-A: JSONL logger — persists traces across sessions ─────────────────────


class JSONLTracer(ObservabilityCallback):
    """Extends ObservabilityCallback to also write every span to a JSONL file."""

    def __init__(self, log_path: str = "./traces.jsonl"):
        super().__init__()
        self.log_path = Path(log_path)

# JSONL (one JSON object per line) is append-friendly and trivially streamable.
    def _write(self, record: dict) -> None:
        """Append a JSON record to the log file."""
        with self.log_path.open("a") as fh:
            fh.write(json.dumps(record) + "\n")

    def on_llm_end(self, response: LLMResult, run_id: Any, **kwargs):
        # Let the parent build the record into self.calls first
        super().on_llm_end(response, run_id, **kwargs)
# Call super() first so self.calls is already populated before we write to disk.
        # Then write the latest completed record to disk
        if self.calls and "error" not in self.calls[-1]:
            record = {
                **self.calls[-1],
                "timestamp": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
            }
            self._write(record)

    def on_llm_error(self, error: Any, run_id: Any, **kwargs):
        super().on_llm_error(error, run_id, **kwargs)
        if self.calls:
            record = {
                **self.calls[-1],
                "timestamp": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
            }
            self._write(record)


# Run a few calls with the JSONL tracer
JSONL_PATH = "./workshop_traces.jsonl"
if Path(JSONL_PATH).exists():
    Path(JSONL_PATH).unlink()  # start fresh for this demo

jsonl_tracer = JSONLTracer(log_path=JSONL_PATH)
llm_j = ChatOpenAI(model="gpt-4o-mini", temperature=0)

for prompt in [
    "Define observability in one sentence.",
    "Define monitoring in one sentence.",
]:
    llm_j.invoke(
        [HumanMessage(content=prompt)],
        config={"callbacks": [jsonl_tracer]},
    )

print(f"Traces written to: {JSONL_PATH}")
print(f"File size: {Path(JSONL_PATH).stat().st_size} bytes\n")

print("=== JSONL contents ===")
with open(JSONL_PATH) as f:
    for line in f:
        record = json.loads(line)
        print(
            f"  run_id={record['run_id'][:8]}...  "
            f"latency={record['latency_ms']}ms  "
            f"tokens={record['total_tokens']}  "
            f"ts={record['timestamp']}"
        )

In [ ]:
# ─── 5-B: Reload and analyse traces from disk ─────────────────────────────────
# Simulate a new session: read the log file and compute stats from scratch.


def load_traces(path: str) -> list[dict]:
    """Load all spans from a JSONL trace file."""
    p = Path(path)
    if not p.exists():
        return []
    return [json.loads(line) for line in p.read_text().splitlines() if line.strip()]


def analyse_traces(traces: list[dict]) -> None:
    """Print a summary report from a list of trace records."""
    ok = [t for t in traces if "error" not in t]
    errors = [t for t in traces if "error" in t]

    print(f"Total spans:  {len(traces)}")
    print(f"Successful:   {len(ok)}")
    print(f"Errors:       {len(errors)}")
    if ok:
        latencies = [t["latency_ms"] for t in ok]
        tokens = [t["total_tokens"] for t in ok]
        print(f"Avg latency:  {sum(latencies) // len(latencies)}ms")
        print(f"Max latency:  {max(latencies)}ms")
        print(f"Total tokens: {sum(tokens)}")
        print(f"Total cost:   ${sum(t.get('cost_usd', 0) for t in ok):.6f}")


traces_from_disk = load_traces(JSONL_PATH)
print(f"Loaded {len(traces_from_disk)} trace(s) from {JSONL_PATH}\n")
analyse_traces(traces_from_disk)

---

## Part 6 — LangSmith Integration

---

LangSmith is LangChain's hosted tracing platform. It automatically captures every trace — LLM calls, chain steps, tool calls, token counts, latency, errors — and displays them in a web UI with search, filtering, and team sharing.

**The key insight:** enabling it requires zero code changes. You set three environment variables and every `invoke()` call is traced automatically.

```bash
# Add to .env to activate LangSmith
LANGCHAIN_TRACING_V2=true
LANGCHAIN_API_KEY=ls__...
LANGCHAIN_PROJECT=my-agent-project
```

### When to use each backend

```
Learning / prototyping  ──▶  Custom callback (no account, no cost)
Team debugging          ──▶  LangSmith (best DX, trace search, prompts hub)
Production infra        ──▶  OpenTelemetry (vendor-neutral, connects to existing stack)
Eval-heavy workflows    ──▶  Arize Phoenix (built-in annotation and scoring)
```

In [ ]:
# ─── 6-A: LangSmith — zero-code integration check ────────────────────────────
# This cell detects whether LangSmith is configured and prints what will happen.
# If LANGSMITH_API_KEY is not set, the agent still runs — tracing is just skipped.

LANGSMITH_API_KEY = os.environ.get("LANGSMITH_API_KEY", "")
LANGCHAIN_TRACING = os.environ.get("LANGCHAIN_TRACING_V2", "false").lower() == "true"
LANGCHAIN_PROJECT = os.environ.get("LANGCHAIN_PROJECT", "default")

print("LangSmith configuration:")
print(f"  LANGCHAIN_TRACING_V2: {LANGCHAIN_TRACING}")
print(f"  LANGCHAIN_API_KEY set: {bool(LANGSMITH_API_KEY)}")
print(f"  LANGCHAIN_PROJECT: {LANGCHAIN_PROJECT}")
print()

if LANGCHAIN_TRACING and LANGSMITH_API_KEY:
    print("LangSmith is ACTIVE.")
    print("Every invoke() call below will be traced automatically.")
    print("View traces at: https://smith.langchain.com")
else:
    print("LangSmith is NOT configured — running without hosted tracing.")
    print()
    print("To enable LangSmith:")
    print("  1. Create a free account at https://smith.langchain.com")
    print("  2. Copy your API key from Settings → API Keys")
    print("  3. Add to .env:")
    print("       LANGCHAIN_TRACING_V2=true")
    print("       LANGCHAIN_API_KEY=ls__...")
    print("       LANGCHAIN_PROJECT=observability-workshop")
    print("  4. Re-run this notebook — no other code changes needed")

In [ ]:
# ─── 6-B: Run the agent — traces go to LangSmith if configured ───────────────
# The code below is IDENTICAL whether or not LangSmith is active.
# That is the point: observability should be a configuration choice, not a code change.


# TypedDict here instead of MessagesState — no message accumulation needed for a one-shot Q&A node.
class SimpleState(TypedDict):
    task: str
    response: str


def create_simple_workflow():
    """One-node graph: LLM answers the task."""
    llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
    graph = StateGraph(SimpleState)

    def llm_node(state: SimpleState) -> SimpleState:
        result = llm.invoke([HumanMessage(content=state["task"])])
        return {"response": result.content}

    graph.add_node("llm", llm_node)
    graph.set_entry_point("llm")
    graph.add_edge("llm", END)
    return graph.compile()
# LangSmith hooks into LangChain's callback system via env vars — no code changes required.


app = create_simple_workflow()

test_tasks = [
    "What is LangSmith used for?",
    "What does OTLP stand for in OpenTelemetry?",
]

for task in test_tasks:
    result = app.invoke({"task": task, "response": ""})
    status = "[LangSmith traced]" if LANGCHAIN_TRACING else "[local only]"
    print(f"Q: {task}")
    print(f"A: {result['response'][:120]} {status}")
    print()

---

## Part 7 — Error Tracking and Resilience

---

Observability is most valuable when things go wrong. The `on_llm_error` hook captures errors without crashing the agent — but you need to be deliberate about:

1. **Recording** the error (not just printing it)
2. **Classifying** errors (rate limit vs network vs model refusal)
3. **Alerting** when error rate exceeds a threshold
4. **Recovering** gracefully so one bad call does not kill the whole trace

### Error Rate SLO

A simple Service Level Objective: error rate < 1% over a rolling window.

```
error_rate = errors / total_calls

error_rate > 0.01  →  alert (Slack, PagerDuty, email)
error_rate > 0.05  →  circuit break (stop sending new requests)
```

In [ ]:
# ─── 7-A: Error injection — see on_llm_error fire safely ─────────────────────
# We subclass ChatOpenAI to throw on the first call, then succeed on the second.


class BrokenLLM(ChatOpenAI):
    """Raises an exception on the first invoke call to simulate an API error."""

    _call_count: int = 0

    def invoke(self, *args, **kwargs):
        self._call_count += 1
        if self._call_count == 1:
            raise RuntimeError("Simulated API timeout: connection reset by peer")
        return super().invoke(*args, **kwargs)


error_tracer = ObservabilityCallback()
broken_llm = BrokenLLM(model="gpt-4o-mini", temperature=0)

# Call 1: will error
try:
    broken_llm.invoke(
        [HumanMessage(content="This will fail.")],
        config={"callbacks": [error_tracer]},
    )
except Exception as e:
    print(f"Call 1 raised (expected): {e}")

# Call 2: will succeed
broken_llm.invoke(
    [HumanMessage(content="What is 2 + 2?")],
    config={"callbacks": [error_tracer]},
)
print("Call 2 succeeded.")

print()
print("=== Error tracer report ===")
for k, v in error_tracer.report().items():
    print(f"  {k}: {v}")

print()
print("=== All call records ===")
for call in error_tracer.calls:
    print(f"  {call}")

rpt = error_tracer.report()
error_rate = rpt["errors"] / max(rpt["total_calls"], 1)
print(f"\nError rate: {error_rate:.0%}")
print("SLO BREACH" if error_rate > 0.01 else "SLO OK")

---

## Exercises

---

Work through these exercises in order. Each one builds on the `ObservabilityCallback` or `FullTracer` from the earlier parts.

### Exercise 1 — Add Tool Tracking

Extend `FullTracer` with `on_tool_start`, `on_tool_end`, and `on_tool_error` methods to time tool calls. Test it by creating a LangGraph agent that uses at least one tool (e.g., a simple Python function wrapped with `@tool`).

**Expected output:** a `tool_spans` list with one record per tool call, each containing `tool_name`, `latency_ms`, and `output_preview`.

### Exercise 2 — File Logger Subclass

Write a `FileTracer` subclass of `JSONLTracer` that:
- Accepts a `session_id` parameter (e.g., `"workshop-2025-06-14"`)
- Includes `session_id` in every trace record
- Writes to `./traces/{session_id}.jsonl` (creating the directory if it does not exist)

**Expected output:** a file at `./traces/workshop-2025-06-14.jsonl` with properly labelled records.

### Exercise 3 — Matplotlib Dashboard

Run `DEMO_TASKS` through the two-node workflow (6 LLM calls total). Then use `matplotlib` to plot:
1. A bar chart of `latency_ms` per call
2. A stacked bar chart of `prompt_tokens` vs `completion_tokens` per call

**Hint:** `cb.calls` is a plain list of dicts — extract any field with a list comprehension and pass directly to `ax.bar()`.

### Exercise 4 — Overhead Measurement

Measure how much overhead the callback adds:
1. Run the same prompt 5 times **without** a callback and record wall-clock time per call
2. Run the same prompt 5 times **with** `ObservabilityCallback` attached
3. Compare average latencies and compute overhead as a percentage of network RTT

**Expected result:** overhead < 1ms per call (pure Python dict operations, no I/O).

In [ ]:
# ── Exercise 1 Starter — Add Tool Tracking ────────────────────────────────────
from langchain_core.tools import tool


# @tool converts the function + docstring into a StructuredTool the LLM can call.
@tool
def get_word_count(text: str) -> int:
    """Count the number of words in a string."""
    return len(text.split())


class ToolTracer(FullTracer):
    def __init__(self):
        super().__init__()
        self.tool_spans: list[dict] = []
        self._tool_starts: dict[str, tuple[float, str]] = {}

    def on_tool_start(
# on_tool_start fires BEFORE execution — record start time here, not in on_tool_end.
        self, serialized: dict, input_str: str, run_id: Any, **kwargs
    ):
        # TODO: record the start time and tool name keyed by str(run_id)
        # Hint: tool_name = serialized.get("name", "unknown_tool")
        pass

    def on_tool_end(self, output: str, run_id: Any, **kwargs):
        # TODO: pop the start, compute latency, append to self.tool_spans
        pass

    def on_tool_error(self, error: Any, run_id: Any, **kwargs):
        # TODO: pop the start, record the error in self.tool_spans
        pass


print("TODO: implement on_tool_start, on_tool_end, on_tool_error")
print("Then create a LangGraph agent that calls get_word_count and attach ToolTracer.")

In [ ]:
# ── Exercise 2 Starter — File Logger Subclass ─────────────────────────────────


class FileTracer(JSONLTracer):
    def __init__(self, session_id: str):
        # TODO: create ./traces/ directory if needed
        # TODO: set log_path to ./traces/{session_id}.jsonl
        # TODO: include session_id in every record via _write()
        super().__init__(log_path="./traces_placeholder.jsonl")
        self.session_id = session_id
        raise NotImplementedError("Complete the __init__ and override _write()")


# Once implemented, uncomment and verify:
# ft = FileTracer(session_id="workshop-2025-06-14")
# llm.invoke([HumanMessage(content="test")], config={"callbacks": [ft]})
# print(Path("./traces/workshop-2025-06-14.jsonl").read_text())

In [ ]:
# ── Exercise 3 Starter — Matplotlib Dashboard ─────────────────────────────────
# Run the two-node workflow 3 times (6 LLM calls), then build charts.

# TODO: uncomment and run after implementing create_two_node_workflow above
#
# import matplotlib.pyplot as plt
#
# dash_tracer = FullTracer()
# dash_app = create_two_node_workflow(dash_tracer)
# for task in DEMO_TASKS:
#     dash_app.invoke({"task": task, "draft": "", "response": ""})
#
# calls = dash_tracer.calls
# indices = list(range(len(calls)))
#
# fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
#
# # Chart 1: latency per call
# ax1.bar(indices, [c["latency_ms"] for c in calls], color="steelblue")
# ax1.set_title("Latency per LLM call (ms)")
# ax1.set_xlabel("Call index")
# ax1.set_ylabel("ms")
#
# # Chart 2: stacked token bars
# prompt_tok = [c["prompt_tokens"] for c in calls]
# completion_tok = [c["completion_tokens"] for c in calls]
# ax2.bar(indices, prompt_tok, label="Prompt", color="#4c72b0")
# ax2.bar(indices, completion_tok, bottom=prompt_tok, label="Completion", color="#dd8452")
# ax2.set_title("Token usage per call")
# ax2.set_xlabel("Call index")
# ax2.set_ylabel("Tokens")
# ax2.legend()
#
# plt.tight_layout()
# plt.savefig("./observability_dashboard.png", dpi=120)
# plt.show()

print("Uncomment the code above to generate the dashboard after running the workflow.")

In [ ]:
# ── Exercise 4 Starter — Overhead Measurement ─────────────────────────────────
OVERHEAD_PROMPT = "What is 2 + 2?"
N_REPS = 5  # keep low during development; use 10+ for accurate measurement

overhead_llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

# Without callback
latencies_no_cb: list[float] = []
for _ in range(N_REPS):
    t0 = time.time()
    overhead_llm.invoke([HumanMessage(content=OVERHEAD_PROMPT)])
    latencies_no_cb.append((time.time() - t0) * 1000)

# With callback
latencies_with_cb: list[float] = []
overhead_cb = ObservabilityCallback()
for _ in range(N_REPS):
    t0 = time.time()
    overhead_llm.invoke(
        [HumanMessage(content=OVERHEAD_PROMPT)],
        config={"callbacks": [overhead_cb]},
    )
    latencies_with_cb.append((time.time() - t0) * 1000)

avg_no_cb = sum(latencies_no_cb) / len(latencies_no_cb)
avg_with_cb = sum(latencies_with_cb) / len(latencies_with_cb)
overhead_ms = avg_with_cb - avg_no_cb

print(f"Runs per condition:     {N_REPS}")
print(f"Avg latency (no cb):    {avg_no_cb:.1f}ms")
print(f"Avg latency (with cb):  {avg_with_cb:.1f}ms")
print(f"Callback overhead:      {overhead_ms:.2f}ms")
print()
pct = abs(overhead_ms) / avg_no_cb * 100
print(f"Overhead as % of network latency: {pct:.1f}%")
print("(Expected: < 1ms, < 0.2% — Python dict operations are negligible vs. network RTT)")

---

## Answer Key

Use this section as a reference after attempting the exercises yourself. These are sample solutions, not the only correct approaches.

---

### Exercise 1 — Add Tool Tracking

**Key insight:** `on_tool_start` receives `serialized["name"]` as the tool name. The pattern is identical to LLM span tracking: store `(time.time(), tool_name)` in `_tool_starts[str(run_id)]`, pop it in `on_tool_end`, compute elapsed ms, append to `tool_spans`.

**What good output looks like:**
```python
tool_tracer.tool_spans
# [{"run_id": "abc...", "tool_name": "get_word_count", "latency_ms": 0, "output_preview": "5"}]
```

Tool calls are almost always sub-millisecond for pure-Python tools. If you see latencies above 100ms, the tool is doing I/O (HTTP, DB) — that is worth separating from LLM latency in your dashboards.

In [ ]:
# Sample solution for Exercise 1


class ToolTracerSolution(FullTracer):
    def __init__(self):
        super().__init__()
        self.tool_spans: list[dict] = []
        self._tool_starts: dict[str, tuple[float, str]] = {}

    def on_tool_start(
        self, serialized: dict, input_str: str, run_id: Any, **kwargs
    ):
        tool_name = serialized.get("name", "unknown_tool")
        self._tool_starts[str(run_id)] = (time.time(), tool_name)

    def on_tool_end(self, output: str, run_id: Any, **kwargs):
        start_time, tool_name = self._tool_starts.pop(str(run_id), (time.time(), "unknown"))
        self.tool_spans.append(
            {
                "run_id": str(run_id),
                "tool_name": tool_name,
                "latency_ms": round((time.time() - start_time) * 1000),
                "output_preview": str(output)[:80],
            }
        )

    def on_tool_error(self, error: Any, run_id: Any, **kwargs):
        self._tool_starts.pop(str(run_id), None)
        self.tool_spans.append({"run_id": str(run_id), "error": str(error)})


print("ToolTracerSolution tracks LLM spans, chain spans, AND tool spans.")
print("To test: create a ReAct agent, attach ToolTracerSolution, invoke with a tool-calling prompt.")

### Exercise 2 — File Logger Subclass

**Key insight:** `Path.mkdir(parents=True, exist_ok=True)` handles the directory creation safely. Override `_write()` to inject `session_id` into every record before delegating to the parent.

**What good output looks like:**
```
./traces/workshop-2025-06-14.jsonl
{"run_id": "...", "session_id": "workshop-2025-06-14", "latency_ms": 820, "total_tokens": 94, ...}
```

In [ ]:
# Sample solution for Exercise 2


class FileTracerSolution(JSONLTracer):
    def __init__(self, session_id: str):
        log_dir = Path("./traces")
        log_dir.mkdir(parents=True, exist_ok=True)
        super().__init__(log_path=str(log_dir / f"{session_id}.jsonl"))
        self.session_id = session_id

    def _write(self, record: dict) -> None:
        """Inject session_id before writing to disk."""
        super()._write({**record, "session_id": self.session_id})


# Quick verification
ft = FileTracerSolution(session_id="workshop-demo")
demo_llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
demo_llm.invoke(
    [HumanMessage(content="What is 1 + 1?")],
    config={"callbacks": [ft]},
)

trace_file = Path("./traces/workshop-demo.jsonl")
if trace_file.exists():
    record = json.loads(trace_file.read_text().splitlines()[0])
    print(f"session_id in record: {record.get('session_id')}")
    print(f"log path: {ft.log_path}")
    print(f"full record keys: {list(record.keys())}")

### Exercise 3 — Matplotlib Dashboard

**Sample answer:** See the commented code in the exercise starter (cell 27). The key is that `cb.calls` is a plain list of dicts — extract any field with a list comprehension and pass it directly to `ax.bar()`.

**What to look for in the charts:** Node 1 (`draft`) typically uses fewer completion tokens than Node 2 (`refine`) because the refinement prompt is longer (it includes the full draft). Latency is usually dominated by network RTT, not token count — a 10-token response can take as long as a 200-token one if the model streams slowly.

### Exercise 4 — Overhead Measurement

**Expected result:** Callback overhead should be under 1ms per call. The callback runs synchronously in Python, performs only dict operations and `time.time()`, and has no I/O. The dominant latency is always the OpenAI API network round-trip (typically 400–1500ms).

**If you see overhead above 5ms:** check whether a `JSONLTracer` or `FileTracerSolution` is also active — disk writes add measurable latency, especially on network filesystems or spinning disks. Separate your in-memory tracer from your file logger in production.

---

## What's Next?

You now have a complete observability toolkit for LangGraph agents. Here is where to go deeper:

### Evaluate the outputs you have been tracing
- **Example 29 — LLM Judge Harness** ([`../29-llm-judge-harness/`](../29-llm-judge-harness/)): use an LLM-as-judge to score the *quality* of the agent outputs you have been timing. Observability gives you latency and tokens; evaluation gives you correctness and relevance. Use both together for a complete picture.
- **Example 16 — RAG Evaluation** ([`../16-rag-eval-notebook/rag_eval_workbook.ipynb`](../16-rag-eval-notebook/rag_eval_workbook.ipynb)): RAGAS metrics (faithfulness, answer relevancy, context precision) — trace token cost alongside quality scores to understand the cost-quality tradeoff.

### Add a hosted backend
- **LangSmith** (https://smith.langchain.com) — zero-code integration, free tier, best developer experience for LangChain/LangGraph projects. Enable with 3 env vars as shown in Part 6.
- **Arize Phoenix** (`pip install arize-phoenix`) — local web UI with built-in span annotation and eval scoring. Ideal for teams that cannot use cloud services.
- **OpenTelemetry + Grafana** — vendor-neutral. Use `opentelemetry-sdk` + `opentelemetry-exporter-otlp` to push spans to any OTEL-compatible backend.

### Scale up
- Add **async** support: `BaseCallbackHandler` hooks are already async-compatible — your subclass works with `await chain.ainvoke()` without changes
- Add **sampling**: in high-traffic production, trace 10% of requests rather than all of them
- Add **alerting**: wire the `report()` method to a Slack webhook when `error_rate > 0.01`

---

### Academic and Technical References

> OpenTelemetry Authors. (2024). *OpenTelemetry Specification — Traces.* CNCF. https://opentelemetry.io/docs/concepts/signals/traces/  
> LangChain. (2024). *Callbacks conceptual guide.* https://python.langchain.com/docs/concepts/callbacks/  
> LangSmith. (2024). *Observability overview.* https://docs.smith.langchain.com/observability/  
> Kleppmann, M. (2017). *Designing Data-Intensive Applications.* O'Reilly. Chapter 12 (distributed tracing concepts).  
> Traceloop. (2024). *OpenLLMetry — OpenTelemetry for LLMs.* https://github.com/traceloop/openllmetry  
> Burns, B., Oppenheimer, D., et al. (2016). *Borg, Omega, and Kubernetes.* ACM Queue 14(1). (foundational work on cluster observability)  

---

*Workshop complete. The next step is example 29 — put a quality score on every trace you just collected.*